# Manual-calibrated ER/MR/LR detection

This notebook treats `outputs/annotations/manual_er_mr_lr-annot.fif` as immutable calibration data. It measures the lowercase manual labels, calibrates a robust detector, opens a separate editable review copy, and saves lowercase `*_auto` results.

In [ ]:
%matplotlib qt
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from emg_lr_segmentation.manual_calibrated_er_mr_lr import (
    AUTO_LABELS, atomic_save_annotations, build_calibration,
    candidates_to_annotations, detect_components, evaluate_candidates,
    extract_annotation_metrics, load_mat_raw, normalized_review_annotations,
    plot_best_epochs, plot_metric_distributions, review_annotations,
    save_calibration,
)

mne.viz.set_browser_backend("qt")

mat_path = Path("my_path/my_dir/my_recording.mat")
manual_annotation_path = project_root / "outputs" / "annotations" / "manual_er_mr_lr-annot.fif"
metrics_dir = project_root / "outputs" / "metrics"
figures_dir = project_root / "outputs" / "figures"
annotations_dir = project_root / "outputs" / "annotations"
for directory in [metrics_dir, figures_dir, annotations_dir]:
    directory.mkdir(parents=True, exist_ok=True)

channel = "GM Right"
baseline_window_ms = (-25.0, -5.0)
browser_scaling = 8.0779357 / 1.5  # 1.5x larger traces than the current MAT browser
skip_interactive_review = os.environ.get("EMG_SKIP_INTERACTIVE_QC", "0") == "1"

In [ ]:
# Load the recording and immutable manual calibration annotations.
if not mat_path.exists():
    raise FileNotFoundError(mat_path)
if not manual_annotation_path.exists():
    raise FileNotFoundError(manual_annotation_path)

raw = load_mat_raw(mat_path)
manual_annotations = mne.read_annotations(manual_annotation_path)
manual_counts = pd.Series(manual_annotations.description, dtype=str).value_counts()
required = {"Stimulus_Auto", "er", "mr", "lr"}
missing = required - set(manual_counts.index)
if missing:
    raise ValueError(f"Missing required annotations: {sorted(missing)}")

print("Recording duration, s:", raw.times[-1])
print("Sampling frequency, Hz:", raw.info["sfreq"])
display(manual_counts.rename("count").to_frame())

In [ ]:
# Measure manual ER/MR/LR waves; point annotations receive inferred bounds.
manual_metrics, manual_polarities, stimulus_times_s = extract_annotation_metrics(
    raw, manual_annotations, channel=channel, baseline_window_ms=baseline_window_ms
)
manual_metrics.to_csv(metrics_dir / "manual_er_mr_lr_metrics.csv", index=False)

quality_summary = (
    manual_metrics.groupby("label")
    .agg(
        total=("label", "size"),
        inferred_bounds=("bounds_inferred", "sum"),
        included=("included_in_calibration", "sum"),
        median_peak_latency_ms=("peak_latency_ms", "median"),
        median_duration_ms=("duration_ms", "median"),
        median_p2p=("p2p_amplitude", "median"),
    )
)
print("Dominant polarities:", manual_polarities)
display(quality_summary)
display(manual_metrics.loc[~manual_metrics.included_in_calibration, ["label", "stimulus_index", "exclusion_reason"]])

In [ ]:
# Median + scaled-MAD calibration and distributions.
calibration, calibration_summary = build_calibration(manual_metrics, manual_polarities)
calibration_summary.to_csv(metrics_dir / "manual_er_mr_lr_calibration_summary.csv", index=False)
save_calibration(calibration, metrics_dir / "manual_er_mr_lr_calibration.json")

plot_metric_distributions(
    manual_metrics, calibration, figures_dir / "manual_er_mr_lr_metric_distributions.png"
)
plt.show()
display(calibration_summary)

In [ ]:
# Run the permissive sequential ER -> MR -> 1-3 LR detector.
auto_candidates = detect_components(
    raw, stimulus_times_s, calibration, channel=channel, baseline_window_ms=baseline_window_ms
)
if auto_candidates.empty:
    raise RuntimeError("The calibrated detector found no complete ER/MR/LR sequences")
auto_candidates.to_csv(metrics_dir / "manual_calibrated_candidates_before_review.csv", index=False)

in_sample_performance = evaluate_candidates(manual_metrics, auto_candidates, calibration)
in_sample_performance.to_csv(metrics_dir / "manual_calibrated_in_sample_performance.csv", index=False)
print("Calibration-set performance (not independent validation):")
display(in_sample_performance)
display(pd.Series(auto_candidates.auto_label, dtype=str).value_counts().rename("count").to_frame())

In [ ]:
# Full interactive review: delete, add, move, or resize lowercase *_auto spans.
# When adding an annotation, use exactly er_auto, mr_auto, or lr_auto.
if skip_interactive_review:
    print("Interactive review skipped; all candidates are retained.")
    descriptions = np.asarray(manual_annotations.description, dtype=str)
    keep = (descriptions == "Stimulus_Auto") | np.char.startswith(descriptions, "BAD") | np.char.startswith(descriptions, "EDGE")
    reviewed_annotations = manual_annotations[keep] + candidates_to_annotations(auto_candidates)
else:
    print("Review candidates, then close the browser to continue.")
    reviewed_annotations = review_annotations(
        raw, manual_annotations, auto_candidates, browser_scaling=browser_scaling, block=True
    )

In [ ]:
# Normalize added point marks, save reviewed metrics, and atomically save annotations.
final_annotations, reviewed_metrics = normalized_review_annotations(
    raw, reviewed_annotations, channel=channel
)
reviewed_metrics.to_csv(metrics_dir / "manual_calibrated_candidates_after_review.csv", index=False)

final_annotation_path = annotations_dir / "stimulus_er_mr_lr_manual_calibrated_auto-annot.fif"
saved_counts = atomic_save_annotations(final_annotations, final_annotation_path)
print("Saved:", final_annotation_path)
print("Round-trip validated counts:", saved_counts)

In [ ]:
# Separate best-10 reviewed epochs, ranked by LR prominence then amplitude.
import importlib
import emg_lr_segmentation.manual_calibrated_er_mr_lr as manual_helpers

manual_helpers = importlib.reload(manual_helpers)
plot_best_epochs = manual_helpers.plot_best_epochs

best_figure, best_10 = plot_best_epochs(
    raw, reviewed_metrics, stimulus_times_s=stimulus_times_s,
    output_path=figures_dir / "manual_calibrated_best_10_epochs.png",
    channel=channel, n_plots=10, time_scale=0.50,
    row_height=2.4, right_edge_padding_ms=0.5,
)
best_10.to_csv(metrics_dir / "manual_calibrated_best_10_epochs.csv", index=False)
display(best_10)
if best_figure is not None:
    plt.show()
else:
    print("No reviewed LR annotations are available for the best-epochs plot.")